In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import math

In [65]:
np.random.seed(42)

days = 365
beds = 75
max_days_padded = days + 100


def generate_arrivals(t, ward):
    if ward == 'A':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        return np.random.poisson(max(0, lambda_A))
    if ward == 'B':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        lambda_B = (1/5) * lambda_A
        return np.random.poisson(max(0, lambda_B))
    if ward == 'C':
        return np.random.poisson(6)
        
def LOS(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = np.log(4 * np.sqrt(2))
    elif ward == 'B':
        mu = np.log(6 * np.sqrt(2))
    elif ward == 'C':
        mu = np.log(5 * np.sqrt(2))
        
    los_days = np.ceil(np.random.lognormal(mean=mu, sigma=sigma, size=arrivals)).astype(int)
    return np.maximum(1, los_days)



def simulate_hospital(generate_arrivals, LOS, beds_A_alloc, beds_B_alloc, beds_C_alloc):
    blocking_fraction = []
    beds_A = np.full(max_days_padded, beds_A_alloc)
    beds_B = np.full(max_days_padded, beds_B_alloc)
    beds_C = np.full(max_days_padded, beds_C_alloc)

    total_arrivals = {'A': 0, 'B': 0, 'C': 0}
    blocked_counts = {'A': 0, 'B': 0, 'C': 0}
    
    utilization_A = []
    utilization_B = []
    utilization_C = []

    for t in range(0,days):
        blocked = 0
        arrivals_A = generate_arrivals(t,'A')
        arrivals_B = generate_arrivals(t,'B')
        arrivals_C = generate_arrivals(t,'C')

        total_arrivals['A'] += arrivals_A
        total_arrivals['B'] += arrivals_B
        total_arrivals['C'] += arrivals_C

        los_A = LOS('A', arrivals_A)
        los_B = LOS('B', arrivals_B)
        los_C = LOS('C', arrivals_C)

        for length in los_A:
            if np.all(beds_A[t:t+length] > 0):
                beds_A[t:t+length] -= 1
            else:
                blocked_counts['A'] += 1


        for length in los_B:
            if np.all(beds_B[t:t+length] > 0):
                beds_B[t:t+length] -= 1
            else:
                if np.all(beds_A[t:t+length] > 0):
                    beds_A[t:t+length] -= 1
                else:
                    blocked_counts['B'] += 1

        for length in los_C:
            if np.all(beds_C[t:t+length] > 0):
                beds_C[t:t+length] -= 1
            else:
                blocked_counts['C'] += 1
        
        utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
        utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)
        utilization_C.append((beds_C_alloc - beds_C[t]) / beds_C_alloc)
                
    total_blocked = sum(blocked_counts.values())
    blocked_frac = total_blocked/sum(total_arrivals.values())
    blocked_ward_A = blocked_counts['A']/total_arrivals['A']
    blocked_ward_B = blocked_counts['B']/total_arrivals['B']
    blocked_ward_C = blocked_counts['C']/total_arrivals['C']

    return total_blocked, blocked_frac, blocked_ward_A, blocked_ward_B, blocked_ward_C


In [66]:
b_A = 20
b_B = 30
b_C = 25
simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)

(2726, 0.5689835107493216, 0.6421296296296296, 0.0, 0.6058823529411764)

In [67]:
best_config = None
min_relocated = float('inf')
results_log = []

step = 1
n_reps = 5  

for b_A in range(1, 75, step):
    for b_B in range(1, 75 - b_A, step):
        b_C = 75 - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        sim_frac = []
        ward_A = []
        ward_B = []
        ward_C = []
        for rep in range(n_reps):
            sim_blocked, sim_blocked_frac, sim_ward_A, sim_ward_B, sim_ward_C = simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            sim_frac.append(sim_blocked_frac)
            ward_A.append(sim_ward_A)
            ward_B.append(sim_ward_B)
            ward_C.append(sim_ward_C)
            
        avg_blocked = np.mean(rep_blocked)
        avg_frac = np.mean(sim_frac)
        avg_A = np.mean(ward_A)
        avg_B = np.mean(ward_B)
        avg_C = np.mean(ward_C)

        results_log.append(((b_A, b_B, b_C), avg_blocked, avg_frac, ward_A, ward_B, ward_C))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)

print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")
print(f"Blocked fraction total: {avg_frac:.2f}")
print(f"Blocked frac from ward A: {avg_A:.2f}")
print(f"Blocked frac from ward B: {avg_B:.2f}")
print(f"Blocked frac from ward C: {avg_C:.2f}")




--- Optimization Complete ---
Optimal Bed Distribution (Ward A, Ward B, Ward C): (35, 2, 38)
Minimum expected relocated patients: 2103.40
Blocked fraction total: 0.50
Blocked frac from ward A: 0.13
Blocked frac from ward B: 0.44
Blocked frac from ward C: 0.91


In [49]:
np.random.seed(42)
best_config = None
min_relocated = float('inf')
results_log = []

step = 1
n_reps = 5  

for b_A in range(5, 75, step):
    for b_B in range(5, 75 - b_A, step):
        b_C = 75 - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        for rep in range(n_reps):
            sim_blocked = simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            
        avg_blocked = np.mean(rep_blocked)
        results_log.append(((b_A, b_B, b_C), avg_blocked))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)

print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")


--- Optimization Complete ---
Optimal Bed Distribution (Ward A, Ward B, Ward C): (27, 7, 41)
Minimum expected relocated patients: 2127.00


In [56]:
np.random.seed(42)
best_config = None
min_relocated = float('inf')
results_log = []

step = 2
n_reps = 5  

for b_A in range(0, 75, step):
    for b_B in range(0, 75 - b_A, step):
        b_C = 75 - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        for rep in range(n_reps):
            sim_blocked = simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            
        avg_blocked = np.mean(rep_blocked)
        results_log.append(((b_A, b_B, b_C), avg_blocked))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)

print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")

/var/folders/wc/y7fy42492mlcbk3r1f8fn_6m0000gn/T/ipykernel_52684/701763795.py:85: RuntimeWarning: invalid value encountered in scalar divide
  utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
/var/folders/wc/y7fy42492mlcbk3r1f8fn_6m0000gn/T/ipykernel_52684/701763795.py:86: RuntimeWarning: invalid value encountered in scalar divide
  utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)


TypeError: 'int' object is not subscriptable

# Sensitivity

In [72]:
np.random.seed(42)

days = 365
beds = 75
max_days_padded = days + 100


def generate_arrivals(t, ward):
    if ward == 'A':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        return np.random.poisson(max(0, lambda_A))
    if ward == 'B':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        lambda_B = (1/5) * lambda_A
        return np.random.poisson(max(0, lambda_B))
    if ward == 'C':
        return np.random.poisson(6)
        
def LOS(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = 8
    elif ward == 'B':
        mu = 12
    elif ward == 'C':
        mu = 10
    
    los_days = np.ceil(np.random.exponential(scale=mu, size=arrivals)).astype(int)
    return np.maximum(1, los_days)



def simulate_hospital(generate_arrivals, LOS, beds_A_alloc, beds_B_alloc, beds_C_alloc):
    blocking_fraction = []
    beds_A = np.full(max_days_padded, beds_A_alloc)
    beds_B = np.full(max_days_padded, beds_B_alloc)
    beds_C = np.full(max_days_padded, beds_C_alloc)

    total_arrivals = {'A': 0, 'B': 0, 'C': 0}
    blocked_counts = {'A': 0, 'B': 0, 'C': 0}
    
    utilization_A = []
    utilization_B = []
    utilization_C = []

    for t in range(0,days):
        blocked = 0
        arrivals_A = generate_arrivals(t,'A')
        arrivals_B = generate_arrivals(t,'B')
        arrivals_C = generate_arrivals(t,'C')

        total_arrivals['A'] += arrivals_A
        total_arrivals['B'] += arrivals_B
        total_arrivals['C'] += arrivals_C

        los_A = LOS('A', arrivals_A)
        los_B = LOS('B', arrivals_B)
        los_C = LOS('C', arrivals_C)

        for length in los_A:
            if np.all(beds_A[t:t+length] > 0):
                beds_A[t:t+length] -= 1
            else:
                blocked_counts['A'] += 1


        for length in los_B:
            if np.all(beds_B[t:t+length] > 0):
                beds_B[t:t+length] -= 1
            else:
                if np.all(beds_A[t:t+length] > 0):
                    beds_A[t:t+length] -= 1
                else:
                    blocked_counts['B'] += 1

        for length in los_C:
            if np.all(beds_C[t:t+length] > 0):
                beds_C[t:t+length] -= 1
            else:
                blocked_counts['C'] += 1
        
        utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
        utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)
        utilization_C.append((beds_C_alloc - beds_C[t]) / beds_C_alloc)
                
    total_blocked = sum(blocked_counts.values())
    blocked_frac = total_blocked/sum(total_arrivals.values())
    blocked_ward_A = blocked_counts['A']/total_arrivals['A']
    blocked_ward_B = blocked_counts['B']/total_arrivals['B']
    blocked_ward_C = blocked_counts['C']/total_arrivals['C']

    return total_blocked, blocked_frac, blocked_ward_A, blocked_ward_B, blocked_ward_C


In [73]:
b_A = 20
b_B = 30
b_C = 25
simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)

(2664,
 0.5524678556615512,
 0.6097560975609756,
 0.002232142857142857,
 0.6079393398751115)

In [74]:
np.random.seed(42)

days = 365
beds = 100
max_days_padded = days + 100


def generate_arrivals(t, ward):
    if ward == 'A':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        return np.random.poisson(max(0, lambda_A))
    if ward == 'B':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        lambda_B = (1/5) * lambda_A
        return np.random.poisson(max(0, lambda_B))
    if ward == 'C':
        return np.random.poisson(6)
        
def LOS(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = np.log(4 * np.sqrt(2))
    elif ward == 'B':
        mu = np.log(6 * np.sqrt(2))
    elif ward == 'C':
        mu = np.log(5 * np.sqrt(2))
        
    los_days = np.ceil(np.random.lognormal(mean=mu, sigma=sigma, size=arrivals)).astype(int)
    return np.maximum(1, los_days)



def simulate_hospital(generate_arrivals, LOS, beds_A_alloc, beds_B_alloc, beds_C_alloc):
    blocking_fraction = []
    beds_A = np.full(max_days_padded, beds_A_alloc)
    beds_B = np.full(max_days_padded, beds_B_alloc)
    beds_C = np.full(max_days_padded, beds_C_alloc)

    total_arrivals = {'A': 0, 'B': 0, 'C': 0}
    blocked_counts = {'A': 0, 'B': 0, 'C': 0}
    
    utilization_A = []
    utilization_B = []
    utilization_C = []

    for t in range(0,days):
        blocked = 0
        arrivals_A = generate_arrivals(t,'A')
        arrivals_B = generate_arrivals(t,'B')
        arrivals_C = generate_arrivals(t,'C')

        total_arrivals['A'] += arrivals_A
        total_arrivals['B'] += arrivals_B
        total_arrivals['C'] += arrivals_C

        los_A = LOS('A', arrivals_A)
        los_B = LOS('B', arrivals_B)
        los_C = LOS('C', arrivals_C)

        for length in los_A:
            if np.all(beds_A[t:t+length] > 0):
                beds_A[t:t+length] -= 1
            else:
                blocked_counts['A'] += 1


        for length in los_B:
            if np.all(beds_B[t:t+length] > 0):
                beds_B[t:t+length] -= 1
            else:
                if np.all(beds_A[t:t+length] > 0):
                    beds_A[t:t+length] -= 1
                else:
                    blocked_counts['B'] += 1

        for length in los_C:
            if np.all(beds_C[t:t+length] > 0):
                beds_C[t:t+length] -= 1
            else:
                blocked_counts['C'] += 1
        
        utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
        utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)
        utilization_C.append((beds_C_alloc - beds_C[t]) / beds_C_alloc)
                
    total_blocked = sum(blocked_counts.values())
    blocked_frac = total_blocked/sum(total_arrivals.values())
    blocked_ward_A = blocked_counts['A']/total_arrivals['A']
    blocked_ward_B = blocked_counts['B']/total_arrivals['B']
    blocked_ward_C = blocked_counts['C']/total_arrivals['C']

    return total_blocked, blocked_frac, blocked_ward_A, blocked_ward_B, blocked_ward_C


In [75]:
b_A = 20
b_B = 30
b_C = 25
simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)

(2726, 0.5689835107493216, 0.6421296296296296, 0.0, 0.6058823529411764)